# Strategy Comparison - A/B Testing for Researchers

This notebook shows you how to compare multiple trading strategies side-by-side.

**Perfect for**: Testing which strategy works best for your research question.

## What You'll Learn

1. How to run multiple strategies in parallel
2. How to compare performance metrics
3. How to visualize strategy differences
4. How to identify which strategy is best for your needs

## Prerequisites

Complete `01_getting_started.ipynb` first to understand the basics.

---

## Step 1: Setup

In [ ]:
# Add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Import strategy tools
from Strategies.Registry import quick_strategy, list_templates

# Import backtest tools
from Backtest.MinimalBacktest import MinimalBacktest
import numpy as np
import pandas as pd
from datetime import date

# Import visualization tools
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Setup complete!")

---

## Step 2: Create Mock Market Data

We'll use the same mock data for all strategies to ensure fair comparison:

In [ ]:
class SimpleMockMDP:
    """Mock market data provider for demonstration."""
    
    def __init__(self, base_rate=5.0, carry_spread=0.10, seed=42):
        self.base_rate = base_rate
        self.carry_spread = carry_spread
        np.random.seed(seed)
    
    def get_pricer(self, currency, as_of):
        return self
    
    def futures_price(self, contract):
        quarter_map = {'H': 0, 'M': 1, 'U': 2, 'Z': 3}
        quarter_code = contract[-2] if len(contract) >= 2 else 'H'
        quarter = quarter_map.get(quarter_code, 0)
        
        rate = self.base_rate + (quarter * self.carry_spread)
        price = 100.0 - rate
        noise = np.random.normal(0, 0.01)
        
        return price + noise

# Create market data
mdp = SimpleMockMDP(base_rate=5.0, carry_spread=0.10, seed=42)

# Define universe and dates
instruments = ['SFRZ4', 'SFRH5', 'SFRM5', 'SFRU5']
start_date = date(2024, 9, 1)
end_date = date(2024, 12, 1)
dates = pd.date_range(start_date, end_date, freq='W').tolist()
dates = [d.date() if hasattr(d, 'date') else d for d in dates]

print("✓ Market data configured")
print(f"  Instruments: {len(instruments)}")
print(f"  Date range: {start_date} to {end_date}")
print(f"  Rebalance periods: {len(dates)}")

---

## Step 3: Define Strategies to Compare

We'll test three different strategy approaches:

1. **Simple Carry** - Trades based on interest rate differentials
2. **Multi-Signal** - Combines carry and momentum signals
3. **Conservative Carry** - Same as simple carry but with higher risk aversion

In [ ]:
# Define strategy configurations
strategy_configs = {
    'Simple Carry': {
        'template': 'simple_carry',
        'risk_aversion': 1.0,
        'description': 'Basic carry strategy with moderate risk'
    },
    'Multi-Signal': {
        'template': 'multi_signal',
        'risk_aversion': 1.0,
        'description': 'Combines carry and momentum signals'
    },
    'Conservative Carry': {
        'template': 'simple_carry',
        'risk_aversion': 2.5,
        'description': 'Carry strategy with high risk aversion'
    }
}

print("Strategies to Compare:")
print("=" * 60)
for name, config in strategy_configs.items():
    print(f"\n{name}:")
    print(f"  Template: {config['template']}")
    print(f"  Risk Aversion: {config['risk_aversion']}")
    print(f"  Description: {config['description']}")

---

## Step 4: Run All Backtests

Now let's run all three strategies and collect the results:

In [ ]:
results = {}

print("Running backtests...\n")

for name, config in strategy_configs.items():
    print(f"Testing: {name}...")
    
    # Create backtest
    backtest = MinimalBacktest(
        mdp=mdp,
        risk_aversion=config['risk_aversion'],
        long_only=True,
        min_history=5
    )
    
    # Run with same seed for consistency
    np.random.seed(42)
    result = backtest.run(contracts=instruments, dates=dates)
    
    results[name] = result
    print(f"  ✓ Complete - Sharpe: {result.sharpe_ratio:.3f}\n")

print("=" * 60)
print("✓ All backtests complete!")

---

## Step 5: Compare Key Metrics

Let's create a summary table comparing all strategies:

In [ ]:
# Create comparison DataFrame
comparison = pd.DataFrame({
    'Sharpe Ratio': {name: r.sharpe_ratio for name, r in results.items()},
    'IC': {name: r.ic for name, r in results.items()},
    'Total Return': {name: r.total_return for name, r in results.items()},
    'Mean Return': {name: r.returns.mean() for name, r in results.items()},
    'Volatility': {name: r.returns.std() for name, r in results.items()},
    'Max Drawdown': {name: (r.returns.cumsum().cummax() - r.returns.cumsum()).max() for name, r in results.items()}
})

# Format for display
pd.options.display.float_format = '{:.4f}'.format

print("Strategy Comparison Table")
print("=" * 80)
print(comparison)
print()

# Highlight best performer in each category
print("\n📊 Best Performers:")
print(f"  Highest Sharpe: {comparison['Sharpe Ratio'].idxmax()} ({comparison['Sharpe Ratio'].max():.3f})")
print(f"  Highest IC: {comparison['IC'].idxmax()} ({comparison['IC'].max():.3f})")
print(f"  Highest Return: {comparison['Total Return'].idxmax()} ({comparison['Total Return'].max():.2%})")
print(f"  Lowest Volatility: {comparison['Volatility'].idxmin()} ({comparison['Volatility'].min():.4f})")

---

## Step 6: Visualize Performance Comparison

Let's create comprehensive visualizations:

In [ ]:
# Create figure with 4 subplots
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Colors for each strategy
colors = {'Simple Carry': 'steelblue', 'Multi-Signal': 'green', 'Conservative Carry': 'coral'}

# 1. Cumulative Returns Comparison
ax1 = fig.add_subplot(gs[0, :])
for name, result in results.items():
    cumulative = (1 + result.returns).cumprod()
    ax1.plot(cumulative.index, cumulative.values, 
             label=name, linewidth=2.5, color=colors[name], alpha=0.8)
ax1.axhline(y=1.0, color='black', linestyle='--', alpha=0.3, label='Break-even')
ax1.set_title('Cumulative Returns Comparison', fontsize=16, fontweight='bold')
ax1.set_ylabel('Portfolio Value (Growth of $1)', fontsize=12)
ax1.legend(loc='best', fontsize=11, framealpha=0.9)
ax1.grid(True, alpha=0.3)

# 2. Sharpe Ratio Comparison
ax2 = fig.add_subplot(gs[1, 0])
sharpes = [results[name].sharpe_ratio for name in strategy_configs.keys()]
bars = ax2.barh(list(strategy_configs.keys()), sharpes, 
                color=[colors[name] for name in strategy_configs.keys()], alpha=0.7)
ax2.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax2.axvline(x=1.0, color='green', linestyle='--', alpha=0.3, label='Good (>1.0)')
ax2.set_title('Sharpe Ratio Comparison', fontsize=14, fontweight='bold')
ax2.set_xlabel('Sharpe Ratio', fontsize=12)
ax2.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, sharpes)):
    ax2.text(val, bar.get_y() + bar.get_height()/2, f'{val:.3f}', 
             ha='left' if val > 0 else 'right', va='center', fontsize=10, fontweight='bold')

# 3. Total Return Comparison
ax3 = fig.add_subplot(gs[1, 1])
returns = [results[name].total_return * 100 for name in strategy_configs.keys()]
bars = ax3.barh(list(strategy_configs.keys()), returns,
                color=[colors[name] for name in strategy_configs.keys()], alpha=0.7)
ax3.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax3.set_title('Total Return Comparison', fontsize=14, fontweight='bold')
ax3.set_xlabel('Total Return (%)', fontsize=12)
ax3.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, returns)):
    ax3.text(val, bar.get_y() + bar.get_height()/2, f'{val:.2f}%', 
             ha='left' if val > 0 else 'right', va='center', fontsize=10, fontweight='bold')

# 4. Risk-Return Scatter
ax4 = fig.add_subplot(gs[2, 0])
for name, result in results.items():
    vol = result.returns.std() * np.sqrt(52)  # Annualized
    ret = result.returns.mean() * 52  # Annualized
    ax4.scatter(vol, ret, s=300, alpha=0.7, color=colors[name], 
                edgecolors='black', linewidth=2, label=name)
    ax4.annotate(name, (vol, ret), xytext=(10, 10), 
                textcoords='offset points', fontsize=10, fontweight='bold')
ax4.axhline(y=0, color='red', linestyle='--', alpha=0.3)
ax4.axvline(x=0, color='red', linestyle='--', alpha=0.3)
ax4.set_title('Risk-Return Profile', fontsize=14, fontweight='bold')
ax4.set_xlabel('Volatility (Annualized)', fontsize=12)
ax4.set_ylabel('Return (Annualized)', fontsize=12)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.1%}'.format(x)))
ax4.grid(True, alpha=0.3)
ax4.legend(loc='best', fontsize=10)

# 5. Drawdown Comparison
ax5 = fig.add_subplot(gs[2, 1])
for name, result in results.items():
    cumulative = result.returns.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    ax5.plot(drawdown.index, drawdown.values, 
             label=name, linewidth=2, color=colors[name], alpha=0.7)
ax5.fill_between(drawdown.index, drawdown.values, 0, alpha=0.2)
ax5.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax5.set_title('Drawdown Comparison', fontsize=14, fontweight='bold')
ax5.set_ylabel('Drawdown', fontsize=12)
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
ax5.legend(loc='best', fontsize=10)
ax5.grid(True, alpha=0.3)

plt.show()

print("✓ Visualizations complete!")

---

## Step 7: Statistical Comparison

Let's look at some additional statistical measures:

In [ ]:
print("Statistical Comparison")
print("=" * 80)
print()

for name, result in results.items():
    print(f"{name}:")
    print(f"  Win Rate: {(result.returns > 0).sum() / len(result.returns):.1%}")
    print(f"  Best Week: {result.returns.max():.2%}")
    print(f"  Worst Week: {result.returns.min():.2%}")
    print(f"  Average Win: {result.returns[result.returns > 0].mean():.2%}")
    print(f"  Average Loss: {result.returns[result.returns < 0].mean():.2%}")
    
    # Calculate profit factor
    total_gains = result.returns[result.returns > 0].sum()
    total_losses = abs(result.returns[result.returns < 0].sum())
    profit_factor = total_gains / total_losses if total_losses > 0 else np.inf
    print(f"  Profit Factor: {profit_factor:.2f}")
    print()

---

## Step 8: Returns Distribution Analysis

Let's visualize how returns are distributed for each strategy:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx]
    
    # Histogram
    returns_pct = result.returns * 100
    ax.hist(returns_pct, bins=15, alpha=0.7, color=colors[name], edgecolor='black')
    
    # Add vertical line for mean
    mean_ret = returns_pct.mean()
    ax.axvline(mean_ret, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_ret:.2f}%')
    ax.axvline(0, color='black', linestyle='-', alpha=0.3)
    
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('Weekly Return (%)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("  • Wider distribution = higher volatility")
print("  • Distribution centered right of zero = positive expected return")
print("  • Long left tail = risk of large losses")

---

## Step 9: Portfolio Weights Comparison

How do the strategies differ in their allocations?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx]
    
    # Calculate average weights
    avg_weights = result.weights.mean()
    
    # Create bar chart
    bars = ax.bar(avg_weights.index, avg_weights.values, 
                   color=colors[name], alpha=0.7, edgecolor='black')
    
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_ylabel('Average Weight', fontsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.0%}'.format(y)))
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        if abs(height) > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1%}',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("  • Conservative strategy has smaller position sizes")
print("  • Different strategies may favor different contracts")
print("  • Weight distribution shows diversification level")

---

## Step 10: Make a Recommendation

Based on the analysis, let's determine which strategy is best:

In [ ]:
print("Strategy Recommendation")
print("=" * 80)
print()

# Create scoring system
scores = {}
for name in results.keys():
    score = 0
    
    # Points for best Sharpe
    if name == comparison['Sharpe Ratio'].idxmax():
        score += 3
        print(f"✓ {name} has the best risk-adjusted returns (Sharpe)")
    
    # Points for best IC
    if name == comparison['IC'].idxmax():
        score += 2
        print(f"✓ {name} has the best signal quality (IC)")
    
    # Points for best total return
    if name == comparison['Total Return'].idxmax():
        score += 2
        print(f"✓ {name} has the highest total return")
    
    # Points for lowest volatility
    if name == comparison['Volatility'].idxmin():
        score += 1
        print(f"✓ {name} has the lowest volatility")
    
    scores[name] = score

print()
print("Overall Scores:")
for name, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name}: {score} points")

winner = max(scores, key=scores.get)
print()
print("="*80)
print(f"🏆 RECOMMENDATION: {winner}")
print("="*80)
print()
print(f"Rationale:")
print(f"  • Sharpe Ratio: {results[winner].sharpe_ratio:.3f}")
print(f"  • Information Coefficient: {results[winner].ic:.3f}")
print(f"  • Total Return: {results[winner].total_return:.2%}")
print()
print("Note: Your choice may differ based on your risk tolerance and objectives!")

---

## Summary: What You've Learned

Congratulations! You now know how to:

✅ Run multiple strategies in parallel

✅ Create comprehensive comparison tables

✅ Visualize performance differences

✅ Analyze statistical properties of strategies

✅ Make data-driven strategy selection decisions

## Key Takeaways

1. **Use consistent data** across all backtests for fair comparison
2. **Look at multiple metrics**, not just returns
3. **Consider risk-adjusted performance** (Sharpe ratio)
4. **Check drawdowns** - how much can you lose?
5. **Examine portfolio weights** - does allocation make sense?

## Next Steps

1. Try comparing more strategies (modify the `strategy_configs` dict)
2. Test with different market conditions (change mock data parameters)
3. See `03_parameter_tuning.ipynb` to optimize individual strategies
4. See `04_results_analysis.ipynb` for deeper performance analytics

---

## Experiment: Add Your Own Strategy

Try adding a fourth strategy to the comparison:

In [ ]:
# YOUR CODE HERE
# Add a new strategy to strategy_configs and re-run the comparison!
# 
# Example:
# strategy_configs['My Custom Strategy'] = {
#     'template': 'simple_carry',
#     'risk_aversion': 0.5,  # More aggressive
#     'description': 'Aggressive carry strategy'
# }
#
# Then re-run cells starting from "Step 4: Run All Backtests"